# 03 - Explorative Datenanalyse: Finanznachrichten (EODHD News API)

**Ziel:** Finanznachrichten zu unseren Waehrungspaaren laden, erkunden und Sentiment-Daten analysieren.

**Datenquelle:** EODHD News API + Sentiment API

**APIs:**
- News API: `https://eodhd.com/api/news?s=EURUSD.FOREX&fmt=json`
- Sentiment API: `https://eodhd.com/api/sentiments?s=EURUSD.FOREX&fmt=json`

**Hinweis:** Jeder News-Request verbraucht 5 API-Calls. Free Plan = 20 Calls/Tag.

---

## 1. Setup und Imports

In [ ]:
# Bibliotheken importieren
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pathlib
import json
from dotenv import load_dotenv

# Darstellung konfigurieren
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print('Setup erfolgreich!')

## 2. API-Key laden

In [ ]:
# .env Datei laden
for env_path in ['../.env', '.env', str(pathlib.Path.home() / 'Documents/GitHub/datawrangling/.env')]:
    if os.path.exists(env_path):
        load_dotenv(dotenv_path=env_path, override=True)
        print(f'.env gefunden unter: {env_path}')
        break

api_key = os.getenv('EODHD_API_KEY')

if api_key and api_key != 'dein_api_key_hier':
    print(f'API-Key geladen (beginnt mit: {api_key[:4]}...)')
else:
    print('FEHLER: Kein API-Key gefunden!')

## 3. Finanznachrichten laden (News API)

Die News API liefert Artikel mit Titel, Inhalt, Tags und **Sentiment-Scores** (polarity, pos, neg, neu).

**Achtung:** Jeder Request verbraucht 5 API-Calls. Wir laden daher erstmal nur fuer ein Waehrungspaar.

In [ ]:
# Konfiguration
CURRENCY_PAIRS = {
    'EURUSD.FOREX': 'EUR/USD',
    'EURCHF.FOREX': 'EUR/CHF',
    'GBPUSD.FOREX': 'GBP/USD',
}

START_DATE = '2024-01-01'
END_DATE = '2025-12-31'

print(f'Zeitraum: {START_DATE} bis {END_DATE}')
print(f'Waehrungspaare: {list(CURRENCY_PAIRS.values())}')
print(f'\nHinweis: Jeder Request = 5 API-Calls (Free Plan = 20/Tag)')

In [ ]:
def load_news(ticker, start_date, end_date, api_key, limit=300):
    """
    Laedt Finanznachrichten von der EODHD News API.
    Pagination: Die API liefert max. 1000 Artikel pro Request.
    Wir laden in Schritten von 'limit' Artikeln.
    """
    all_articles = []
    offset = 0
    
    while True:
        url = 'https://eodhd.com/api/news'
        params = {
            's': ticker,
            'from': start_date,
            'to': end_date,
            'limit': limit,
            'offset': offset,
            'api_token': api_key,
            'fmt': 'json'
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f'  FEHLER: HTTP {response.status_code}')
            break
        
        articles = response.json()
        
        if not articles:
            break
        
        all_articles.extend(articles)
        print(f'  Offset {offset}: {len(articles)} Artikel geladen')
        
        # Wenn weniger als limit zurueckkommen, sind wir fertig
        if len(articles) < limit:
            break
        
        offset += limit
    
    return all_articles

print('Funktion definiert.')

In [ ]:
# Nachrichten fuer alle Waehrungspaare laden
# ACHTUNG: Verbraucht API-Calls! Bei Free Plan evtl. nur 1 Paar laden.
news_data = {}

for eodhd_symbol, pair_name in CURRENCY_PAIRS.items():
    print(f'\nLade Nachrichten fuer {pair_name} ({eodhd_symbol})...')
    
    articles = load_news(eodhd_symbol, START_DATE, END_DATE, api_key)
    
    if articles:
        # In DataFrame umwandeln
        df = pd.DataFrame(articles)
        
        # Datum parsen
        df['date'] = pd.to_datetime(df['date'])
        df['date_only'] = df['date'].dt.date  # Nur Datum ohne Uhrzeit
        
        # Sentiment-Spalten extrahieren (nested dict -> separate Spalten)
        if 'sentiment' in df.columns:
            sentiment_df = pd.json_normalize(df['sentiment'])
            df = pd.concat([df.drop(columns=['sentiment']), sentiment_df], axis=1)
        
        news_data[pair_name] = df
        print(f'  Gesamt: {len(df)} Artikel geladen')
    else:
        print(f'  Keine Artikel gefunden.')

print('\nAlle Nachrichten geladen!')

## 4. Erste Datenuebersicht

In [ ]:
# Uebersicht
for pair_name, df in news_data.items():
    print(f'\n{"=" * 60}')
    print(f'{pair_name}')
    print(f'{"=" * 60}')
    print(f'Anzahl Artikel: {len(df)}')
    print(f'Zeitraum: {df["date"].min()} bis {df["date"].max()}')
    print(f'Spalten: {list(df.columns)}')
    print(f'\nDatentypen:')
    print(df.dtypes)
    print(f'\nErste 3 Artikel (Titel):')
    for i, row in df.head(3).iterrows():
        print(f'  [{row["date"].strftime("%Y-%m-%d")}] {row["title"][:80]}...')

## 5. Datenqualitaetspruefung

In [ ]:
# Fehlende Werte und Sentiment-Statistiken
for pair_name, df in news_data.items():
    print(f'\n{"=" * 60}')
    print(f'{pair_name} - Datenqualitaet')
    print(f'{"=" * 60}')
    
    # Fehlende Werte
    missing = df.isnull().sum()
    print(f'\nFehlende Werte:')
    for col in df.columns:
        if missing[col] > 0:
            print(f'  {col}: {missing[col]} ({missing[col]/len(df)*100:.1f}%)')
    if missing.sum() == 0:
        print('  Keine fehlenden Werte!')
    
    # Sentiment-Statistiken
    if 'polarity' in df.columns:
        print(f'\nSentiment-Statistiken (polarity):')
        print(f'  Mittelwert: {df["polarity"].mean():.4f}')
        print(f'  Median:     {df["polarity"].median():.4f}')
        print(f'  Std:        {df["polarity"].std():.4f}')
        print(f'  Min:        {df["polarity"].min():.4f}')
        print(f'  Max:        {df["polarity"].max():.4f}')

In [ ]:
# Artikel pro Tag zaehlen
for pair_name, df in news_data.items():
    print(f'\n{pair_name}:')
    daily_count = df.groupby('date_only').size()
    print(f'  Durchschnitt Artikel/Tag: {daily_count.mean():.1f}')
    print(f'  Max Artikel/Tag:          {daily_count.max()}')
    print(f'  Min Artikel/Tag:          {daily_count.min()}')
    print(f'  Tage mit Artikeln:        {len(daily_count)}')

## 6. Visualisierung

### 6.1 Artikel pro Tag

In [ ]:
# Artikel pro Tag plotten
fig, axes = plt.subplots(len(news_data), 1, figsize=(14, 4 * len(news_data)))

if len(news_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, news_data.items()):
    daily_count = df.groupby('date_only').size()
    ax.bar(daily_count.index, daily_count.values, alpha=0.7, width=1)
    ax.set_title(f'Artikel pro Tag: {pair_name}', fontsize=14)
    ax.set_xlabel('Datum')
    ax.set_ylabel('Anzahl Artikel')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Sentiment-Verteilung

In [ ]:
# Sentiment-Verteilung (Polarity)
fig, axes = plt.subplots(1, len(news_data), figsize=(6 * len(news_data), 5))

if len(news_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, news_data.items()):
    if 'polarity' in df.columns:
        ax.hist(df['polarity'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
        ax.set_title(f'Sentiment Polarity: {pair_name}', fontsize=12)
        ax.set_xlabel('Polarity (0=negativ, 1=positiv)')
        ax.set_ylabel('Haeufigkeit')
        ax.axvline(x=df['polarity'].mean(), color='red', linestyle='--', label=f'Mittelwert: {df["polarity"].mean():.3f}')
        ax.legend()

plt.tight_layout()
plt.show()

### 6.3 Durchschnittliches Sentiment pro Tag

In [ ]:
# Durchschnittliches taegliches Sentiment
fig, axes = plt.subplots(len(news_data), 1, figsize=(14, 4 * len(news_data)))

if len(news_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, news_data.items()):
    if 'polarity' in df.columns:
        daily_sentiment = df.groupby('date_only')['polarity'].mean()
        ax.plot(daily_sentiment.index, daily_sentiment.values, linewidth=1, alpha=0.7)
        ax.axhline(y=daily_sentiment.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mittelwert: {daily_sentiment.mean():.3f}')
        ax.set_title(f'Durchschnittliches Sentiment pro Tag: {pair_name}', fontsize=14)
        ax.set_xlabel('Datum')
        ax.set_ylabel('Polarity')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.4 Sentiment-Aufteilung (positiv, negativ, neutral)

In [ ]:
# Sentiment-Komponenten vergleichen
for pair_name, df in news_data.items():
    if all(col in df.columns for col in ['pos', 'neg', 'neu']):
        print(f'\n{pair_name} - Sentiment-Aufteilung (Durchschnitt):')
        print(f'  Positiv:  {df["pos"].mean():.4f}')
        print(f'  Negativ:  {df["neg"].mean():.4f}')
        print(f'  Neutral:  {df["neu"].mean():.4f}')
        
        # Pie Chart
        fig, ax = plt.subplots(figsize=(6, 6))
        values = [df['pos'].mean(), df['neg'].mean(), df['neu'].mean()]
        labels = ['Positiv', 'Negativ', 'Neutral']
        colors = ['#2ecc71', '#e74c3c', '#95a5a6']
        ax.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        ax.set_title(f'Sentiment-Aufteilung: {pair_name}')
        plt.show()

## 7. Rohdaten speichern

In [ ]:
# Rohdaten als CSV und JSON speichern
OUTPUT_DIR = '../data/raw/news/eodhd'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for pair_name, df in news_data.items():
    safe_name = pair_name.replace('/', '_')
    
    # CSV speichern
    csv_filename = f'{safe_name}_news_{START_DATE}_to_{END_DATE}.csv'
    csv_path = os.path.join(OUTPUT_DIR, csv_filename)
    df.to_csv(csv_path, index=False)
    print(f'Gespeichert: {csv_path} ({len(df)} Artikel)')

print('\nAlle Nachrichten-Rohdaten gespeichert!')

## 8. Zusammenfassung

### Erkenntnisse:
- **Anzahl Artikel:** (Ergebnisse eintragen)
- **Sentiment-Verteilung:** (Ergebnisse eintragen)
- **Datenqualitaet:** (Ergebnisse eintragen)
- **Zeitliche Abdeckung:** (Ergebnisse eintragen)

### Naechste Schritte:
1. Zweite Nachrichtenquelle laden (Web Scraping)
2. Sentiment-Analyse mit VADER/FinBERT auf Nachrichtentexte anwenden
3. Sentiment mit Forex-Kursen zusammenfuehren
4. Korrelationsanalyse: Beeinflusst Nachrichtenstimmung die Kurse?